# Build TimeMMD final CSVs with latest text

This notebook merges `dataset/TimeMMD/numerical` and `dataset/TimeMMD/textual` into `dataset/TimeMMD/final`.

For each numeric row, it selects the latest textual row whose `end_date` is not after the numeric row's cutoff date and is still inside a configurable lookback window. If no valid text is found, it writes `No information available`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

BASE_DIR = Path("dataset/TimeMMD")
NUMERICAL_DIR = BASE_DIR / "numerical"
TEXTUAL_DIR = BASE_DIR / "textual"
FINAL_DIR = BASE_DIR / "final"

# Example: 31 days ~= one month. Increase this for slower-moving datasets.
LOOKBACK_DAYS = 31

# If two text rows have the same timestamp, prefer report over search.
TEXT_SOURCE_PRIORITY = ["report", "search"]
FALLBACK_TEXT = "No information available"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Writing merged files to: {FINAL_DIR.resolve()}")


In [ ]:
def clean_text(value):
    """Normalize textual fields and map empty/NA-like values to fallback."""
    if pd.isna(value):
        return FALLBACK_TEXT
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    if not text or text.upper() in {"NA", "N/A", "NAN", "NONE", "NULL"}:
        return FALLBACK_TEXT
    return text


def is_available_text(value):
    return isinstance(value, str) and value.strip() and value != FALLBACK_TEXT


def pick_date_column(df):
    if "date" in df.columns:
        return "date"
    if "Date" in df.columns:
        return "Date"
    raise ValueError("No date column found. Expected 'date' or 'Date'.")


def prepare_numeric_dates(df):
    """Return a numeric dataframe with canonical date/start/end columns for alignment."""
    df = df.copy()
    date_col = pick_date_column(df)
    df["_numeric_date"] = pd.to_datetime(df[date_col], errors="coerce")

    if "start_date" in df.columns:
        df["_numeric_start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
    else:
        df["_numeric_start_date"] = df["_numeric_date"]

    if "end_date" in df.columns:
        df["_numeric_end_date"] = pd.to_datetime(df["end_date"], errors="coerce")
    else:
        df["_numeric_end_date"] = df["_numeric_date"]

    df["_numeric_start_date"] = df["_numeric_start_date"].fillna(df["_numeric_date"])
    df["_numeric_end_date"] = df["_numeric_end_date"].fillna(df["_numeric_date"])

    reversed_rows = df["_numeric_start_date"] > df["_numeric_end_date"]
    if reversed_rows.any():
        starts = df.loc[reversed_rows, "_numeric_start_date"].copy()
        df.loc[reversed_rows, "_numeric_start_date"] = df.loc[reversed_rows, "_numeric_end_date"]
        df.loc[reversed_rows, "_numeric_end_date"] = starts

    df = df.sort_values(
        ["_numeric_start_date", "_numeric_end_date", "_numeric_date"],
        ascending=[True, True, True],
    ).reset_index(drop=True)
    return df


def load_textual_dataset(dataset_name):
    """Load report/search text files for one TimeMMD dataset."""
    pieces = []
    dataset_text_dir = TEXTUAL_DIR / dataset_name

    for source_rank, source in enumerate(TEXT_SOURCE_PRIORITY):
        path = dataset_text_dir / f"{dataset_name}_{source}.csv"
        if not path.exists():
            continue

        df = pd.read_csv(path)
        if not {"start_date", "end_date"}.issubset(df.columns):
            raise ValueError(f"{path} must contain start_date and end_date")

        df = df.copy()
        df["text_source"] = source
        df["_source_rank"] = source_rank
        df["_text_start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
        df["_text_end_date"] = pd.to_datetime(df["end_date"], errors="coerce")

        if "fact" not in df.columns:
            df["fact"] = FALLBACK_TEXT
        if "preds" not in df.columns:
            df["preds"] = FALLBACK_TEXT

        df["fact"] = df["fact"].map(clean_text)
        df["preds"] = df["preds"].map(clean_text)
        df = df.dropna(subset=["_text_start_date", "_text_end_date"])
        reversed_rows = df["_text_start_date"] > df["_text_end_date"]
        if reversed_rows.any():
            starts = df.loc[reversed_rows, "_text_start_date"].copy()
            df.loc[reversed_rows, "_text_start_date"] = df.loc[reversed_rows, "_text_end_date"]
            df.loc[reversed_rows, "_text_end_date"] = starts
        df = df[df["fact"].map(is_available_text)]
        pieces.append(df)

    if not pieces:
        return pd.DataFrame(
            columns=["fact", "preds", "text_source", "_source_rank", "_text_start_date", "_text_end_date"]
        )

    text_df = pd.concat(pieces, ignore_index=True)
    text_df = text_df.sort_values(
        ["_text_end_date", "_text_start_date", "_source_rank"],
        ascending=[True, True, True],
    ).reset_index(drop=True)
    return text_df


In [ ]:
def select_latest_text_for_row(row, text_df, lookback_days=LOOKBACK_DAYS):
    """Select latest non-future text inside [cutoff-lookback, cutoff]."""
    cutoff = row["_numeric_end_date"]
    if pd.isna(cutoff) or text_df.empty:
        return {
            "fact": FALLBACK_TEXT,
            "preds": FALLBACK_TEXT,
            "text_source": "none",
            "text_start_date": pd.NaT,
            "text_end_date": pd.NaT,
            "text_age_days": np.nan,
            "text_window_days": lookback_days,
        }

    window_start = cutoff - pd.Timedelta(days=lookback_days)
    eligible = text_df[
        (text_df["_text_end_date"] <= cutoff)
        & (text_df["_text_end_date"] >= window_start)
    ]

    if eligible.empty:
        return {
            "fact": FALLBACK_TEXT,
            "preds": FALLBACK_TEXT,
            "text_source": "none",
            "text_start_date": pd.NaT,
            "text_end_date": pd.NaT,
            "text_age_days": np.nan,
            "text_window_days": lookback_days,
        }

    # Latest first; tie-break toward report via lower _source_rank.
    chosen = eligible.sort_values(
        ["_text_end_date", "_text_start_date", "_source_rank"],
        ascending=[False, False, True],
    ).iloc[0]

    return {
        "fact": chosen["fact"],
        "preds": chosen["preds"],
        "text_source": chosen["text_source"],
        "text_start_date": chosen["_text_start_date"],
        "text_end_date": chosen["_text_end_date"],
        "text_age_days": int((cutoff - chosen["_text_end_date"]).days),
        "text_window_days": lookback_days,
    }


def merge_one_dataset(dataset_name, lookback_days=LOOKBACK_DAYS):
    numeric_path = NUMERICAL_DIR / dataset_name / f"{dataset_name}.csv"
    if not numeric_path.exists():
        raise FileNotFoundError(numeric_path)

    numeric_df = pd.read_csv(numeric_path)
    numeric_df = prepare_numeric_dates(numeric_df)
    text_df = load_textual_dataset(dataset_name)

    selected = [select_latest_text_for_row(row, text_df, lookback_days) for _, row in numeric_df.iterrows()]
    selected_df = pd.DataFrame(selected)

    if text_df.empty:
        text_span_start = pd.NaT
        text_span_end = pd.NaT
        in_text_span = pd.Series(False, index=numeric_df.index)
    else:
        text_span_start = text_df["_text_start_date"].min()
        text_span_end = text_df["_text_end_date"].max()
        in_text_span = numeric_df["_numeric_end_date"].between(text_span_start, text_span_end)

    output_df = pd.concat(
        [numeric_df.drop(columns=["_numeric_date", "_numeric_start_date", "_numeric_end_date"]), selected_df],
        axis=1,
    )

    # Keep dates readable in CSV.
    for col in ["text_start_date", "text_end_date"]:
        output_df[col] = pd.to_datetime(output_df[col], errors="coerce").dt.strftime("%Y-%m-%d")

    output_dir = FINAL_DIR / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{dataset_name}.csv"
    output_df.to_csv(output_path, index=False)

    with_text_mask = output_df["text_source"] != "none"
    in_span_with_text = with_text_mask & in_text_span.reset_index(drop=True)
    rows_in_text_span = int(in_text_span.sum())

    summary = {
        "dataset": dataset_name,
        "rows": len(output_df),
        "text_span_start": text_span_start.strftime("%Y-%m-%d") if pd.notna(text_span_start) else None,
        "text_span_end": text_span_end.strftime("%Y-%m-%d") if pd.notna(text_span_end) else None,
        "rows_in_text_span": rows_in_text_span,
        "with_text": int(with_text_mask.sum()),
        "with_text_in_span": int(in_span_with_text.sum()),
        "no_text": int((~with_text_mask).sum()),
        "report": int((output_df["text_source"] == "report").sum()),
        "search": int((output_df["text_source"] == "search").sum()),
        "output_path": str(output_path),
    }
    return output_df, summary


In [ ]:
dataset_names = sorted([p.name for p in NUMERICAL_DIR.iterdir() if p.is_dir()])
dataset_names


In [ ]:
summaries = []
for dataset_name in dataset_names:
    _, summary = merge_one_dataset(dataset_name, lookback_days=LOOKBACK_DAYS)
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)
summary_df["overall_coverage"] = summary_df["with_text"] / summary_df["rows"]
summary_df["coverage_in_text_span"] = np.where(
    summary_df["rows_in_text_span"] > 0,
    summary_df["with_text_in_span"] / summary_df["rows_in_text_span"],
    np.nan,
)
summary_df.to_csv(FINAL_DIR / "coverage_summary.csv", index=False)
summary_df


In [ ]:
# Quick inspection: Agriculture often reveals alignment problems fastest.
agri_path = FINAL_DIR / "Agriculture" / "Agriculture.csv"
agri = pd.read_csv(agri_path)
agri[["date", "start_date", "end_date", "OT", "fact", "text_source", "text_end_date", "text_age_days"]].tail(12)
